## Optional file renaming for Olympus acquisitions

If images are exported directly from Olympus acquisition software, filenames often use microscope-specific channel identifiers (e.g., `C001`, `C002`, `C003`). The provided renaming script converts these identifiers into the standardized channel naming used by this pipeline (`ch1`, `ch2`, `ch3`) while preserving the acquisition prefix and Z-slice index.

This step is only required when working with raw Olympus-exported files. If your images are already formatted using the expected naming convention (`prefix_channel_Zxxx.tif`), this step can be skipped. Otherwise, filenames must be manually adjusted to match the required format before running the analysis pipeline.

## Step1: Change the first frame folder to 0000

For Olympus exports, image folders are commonly named as `{name}.tif.frames`, `{name}_0001.tif.frames`, `{name}_0002.tif.frames` etc... To make the acquisition order explicit and compatible with the pipeline, the first folder should be renamed to `0000_{name}.tif.frames` so that all sequential acquisitions follow a consistent indexed format.

## Step2: Rename channels

This script standardizes TIFF filenames across imaging subfolders by converting microscope channel codes (C001–C003) into consistent channel labels (ch1–ch3) and preserving the Z-slice index. Files that do not match the expected naming pattern are skipped.

In [ ]:
import os
import re
import shutil

main_folder = "/Volumes/DAVIDESPERI/BCH/Data analysis/Sensors/AMPK_sensor/In_vivo/Old pipeline/20251221_AMPK_sensor/Mouse1"

channel_map = {
    "C001": "ch1",
    "C002": "ch3",
    "C003": "ch2"
}

for subfolder_name in os.listdir(main_folder):
    subfolder_path = os.path.join(main_folder, subfolder_name)

    if os.path.isdir(subfolder_path):
        print(f"Processing folder: {subfolder_name}")

        for filename in os.listdir(subfolder_path):
            if not filename.lower().endswith(".tif"):
                continue

            old_path = os.path.join(subfolder_path, filename)
            match = re.match(r"(.*?)[_-]?(C00[123])Z(\d+)\.tif", filename)
            if not match:
                print(f"⚠️ Skipped (pattern not matched): {filename}")
                continue

            prefix, channel_code, z_index = match.groups()
            prefix = prefix.rstrip("_-")

            if channel_code not in channel_map:
                print(f"⚠️ Unknown channel: {filename}")
                continue

            new_channel = channel_map[channel_code]
            new_name = f"{prefix}_{new_channel}_Z{z_index}.tif"
            new_path = os.path.join(subfolder_path, new_name)

            os.rename(old_path, new_path)
            print(f"✅ Renamed: {filename} → {new_name}")


Processing folder: ExRai_0001.tif.frames
⚠️ Skipped (pattern not matched): ExRai_0001_ch3_Z009.tif
⚠️ Skipped (pattern not matched): ExRai_0001_ch2_Z009.tif
⚠️ Skipped (pattern not matched): ExRai_0001_ch2_Z008.tif
⚠️ Skipped (pattern not matched): ExRai_0001_ch3_Z008.tif
⚠️ Skipped (pattern not matched): ExRai_0001_ch1_Z014.tif
⚠️ Skipped (pattern not matched): ExRai_0001_ch1_Z015.tif
⚠️ Skipped (pattern not matched): ExRai_0001_ch1_Z001.tif
⚠️ Skipped (pattern not matched): ExRai_0001_ch1_Z017.tif
⚠️ Skipped (pattern not matched): ExRai_0001_ch1_Z003.tif
⚠️ Skipped (pattern not matched): ExRai_0001_ch1_Z002.tif
⚠️ Skipped (pattern not matched): ExRai_0001_ch1_Z016.tif
⚠️ Skipped (pattern not matched): ExRai_0001_ch1_Z012.tif
⚠️ Skipped (pattern not matched): ExRai_0001_ch1_Z006.tif
⚠️ Skipped (pattern not matched): ExRai_0001_ch1_Z007.tif
⚠️ Skipped (pattern not matched): ExRai_0001_ch1_Z013.tif
⚠️ Skipped (pattern not matched): ExRai_0001_ch1_Z005.tif
⚠️ Skipped (pattern not matched

## Step3: Organize 800nm and 920nm into one folder

This script scans frame folders, reads metadata to identify the Channel 2 laser wavelength, and pairs 800 nm and 920 nm acquisitions belonging to the same imaging series. For each matched pair, it creates a view-specific analysis folder, copies the corresponding `ch2` Z-stack images using standardized filenames, and saves the associated metadata files. Unpaired or unrecognized acquisitions are skipped.

In [ ]:
base_path = main_folder
analysis_root = os.path.join(main_folder, "analysis")
os.makedirs(analysis_root, exist_ok=True)

def get_laser_wavelength(frames_folder_path):
    for file in os.listdir(frames_folder_path):
        if file.endswith(".txt"):
            txt_path = os.path.join(frames_folder_path, file)
            with open(txt_path, "r", encoding="utf-8") as f:
                lines = f.readlines()

            in_ch2 = False
            for line in lines:
                if "[Channel 2]" in line:
                    in_ch2 = True
                elif in_ch2 and "Laser Wavelength" in line:
                    match = re.search(r"(\d+)\s*\[nm\]", line)
                    if match:
                        return int(match.group(1)), txt_path
                elif line.startswith("[Channel 3]"):
                    break
    return None, None

frames = []

for f in os.listdir(base_path):
    if f.endswith("tif.frames"):
        match = re.search(r"_(\d{4})\.tif", f)
        if not match:
            print(f"⚠️ Skipped (bad name): {f}")
            continue

        index = int(match.group(1))
        fpath = os.path.join(base_path, f)

        wl, txt_path = get_laser_wavelength(fpath)

        print(f"Detected: {f} → {wl} nm")

        if wl not in [800, 920]:
            print(f"⚠️ Unknown wavelength, skipped: {f}")
            continue

        frames.append((index, f, wl, txt_path))

frames.sort()

buffer = {}
view_counter = 1

for index, folder, wl, txt_path in frames:
    buffer[wl] = {"folder": folder, "txt_path": txt_path}
    print(f"Buffered: {folder} → {wl} nm")

    # Only create view when BOTH channels exist
    if 800 in buffer and 920 in buffer:
        view_folder = os.path.join(analysis_root, f"view{view_counter}")
        os.makedirs(view_folder, exist_ok=True)

        print(f"\n✅ Creating view{view_counter}")
        print(f"   800 → {buffer[800]['folder']}")
        print(f"   920 → {buffer[920]['folder']}")

        for wl_used in [800, 920]:
            src_folder = os.path.join(base_path, buffer[wl_used]["folder"])

            for fname in os.listdir(src_folder):
                z_match = re.search(r"_ch2_(Z\d+\.tif)$", fname)
                if not z_match:
                    continue

                src_path = os.path.join(src_folder, fname)
                z_part = z_match.group(1)
                new_name = f"{wl_used}_ch2_{z_part}"
                dst_path = os.path.join(view_folder, new_name)

                if os.path.exists(dst_path):
                    print(f"⚠️ Overwrite blocked: {dst_path}")
                    continue

                shutil.copyfile(src_path, dst_path)
                print(f"✅ Copied → {new_name}")

        for wl_used in [800, 920]:
            shutil.copyfile(
                buffer[wl_used]["txt_path"],
                os.path.join(view_folder, f"{wl_used}_metadata.txt")
            )
        view_counter += 1
        buffer.clear()


if buffer:
    print("⚠️ Leftover unpaired frame(s) after last view:")
    for wl, info in buffer.items():
        print(f"   {wl} → {info['folder']}")


Detected: ExRai_0001.tif.frames → 800 nm
Detected: ExRai_0000.tif.frames → 920 nm
Buffered: ExRai_0000.tif.frames → 920 nm
Buffered: ExRai_0001.tif.frames → 800 nm

✅ Creating view1
   800 → ExRai_0001.tif.frames
   920 → ExRai_0000.tif.frames
✅ Copied → 800_ch2_Z009.tif
✅ Copied → 800_ch2_Z008.tif
✅ Copied → 800_ch2_Z005.tif
✅ Copied → 800_ch2_Z011.tif
✅ Copied → 800_ch2_Z010.tif
✅ Copied → 800_ch2_Z004.tif
✅ Copied → 800_ch2_Z012.tif
✅ Copied → 800_ch2_Z006.tif
✅ Copied → 800_ch2_Z007.tif
✅ Copied → 800_ch2_Z013.tif
✅ Copied → 800_ch2_Z017.tif
✅ Copied → 800_ch2_Z003.tif
✅ Copied → 800_ch2_Z002.tif
✅ Copied → 800_ch2_Z016.tif
✅ Copied → 800_ch2_Z014.tif
✅ Copied → 800_ch2_Z015.tif
✅ Copied → 800_ch2_Z001.tif
✅ Copied → 920_ch2_Z001.tif
✅ Copied → 920_ch2_Z002.tif
✅ Copied → 920_ch2_Z003.tif
✅ Copied → 920_ch2_Z007.tif
✅ Copied → 920_ch2_Z006.tif
✅ Copied → 920_ch2_Z012.tif
✅ Copied → 920_ch2_Z004.tif
✅ Copied → 920_ch2_Z010.tif
✅ Copied → 920_ch2_Z011.tif
✅ Copied → 920_ch2_Z005.tif


## Can now run the S2_per_view_analysis on the created view folders